# 答案添削メカニズム — Google Colab

手書き数学答案を GLM-OCR で読み、UniMERNet Small で数式を読み直し、Qwen3.8-27B（4bit）で採点します。
結果は Google Drive に残します。GPU は A100 40GB を想定し、割り当てが L4 / T4 のときは自動で設定を落とします。

このノートは上から順に実行します。モデルは同時に載せず、OCR → 数式 → 採点の順に切り替えます。

**A100 40GB では、セットアップ、モデル選択（`unsloth/Qwen3.8-27B-unsloth-bnb-4bit`）、cloudflared の URL 表示まで確認済みです。** 採点の初回は、重みのダウンロードと読み込みに時間がかかります。


## 1. Drive をマウントする

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. リポジトリを取って依存関係を入れる

`BRANCH` の既定は、マージ前の PR ブランチです。リポジトリの既定ブランチ `2026-05-03-aho1` には `scripts/colab_setup.sh` がありません。マージ後は `BRANCH` を `2026-05-03-aho1` に変えるか、実行前に環境変数 `REPO_BRANCH` を置いてください。


In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/GUB-UI/Answer-correction-mechanism.git"
# 既定ブランチには colab_setup.sh が無い。マージされるまで PR ブランチを使う。
BRANCH = os.environ.get("REPO_BRANCH", "cursor/colab-cuda-teacher-compare-77a2")
ROOT = Path("/content/Answer-correction-mechanism")
if not (ROOT / ".git").exists():
    !git clone --branch {BRANCH} --single-branch {REPO} {ROOT}
else:
    # 既定ブランチで clone 済みでも、指定ブランチへ切り替える
    !git -C {ROOT} fetch origin {BRANCH}
    !git -C {ROOT} checkout -B {BRANCH} origin/{BRANCH}
os.chdir(ROOT / "math-answer-diagnosis")
print("checked out", BRANCH)
!git -C {ROOT} log -1 --oneline

# セッションが切れても答案・採点・モデルキャッシュが残る場所
os.environ["APP_DATA_DIR"] = "/content/drive/MyDrive/answer-correction"
os.environ["HF_HOME"] = os.environ["APP_DATA_DIR"] + "/hf-cache"
os.environ["UNIMERNET_MODEL_DIR"] = os.environ["APP_DATA_DIR"] + "/models/unimernet_small"
os.environ["APP_CONFIG_FILE"] = "config.colab.yaml"
os.environ["SERVICE_MODE"] = "managed"
# トンネルは公開されるので、必ず自分だけの文字列に変える
os.environ["APP_PASSWORD"] = "change-me"

!bash scripts/colab_setup.sh


## 3. GPU に合わせてモデルを決める

| 割り当て | 診断モデル |
| --- | --- |
| A100 40GB 級 | `unsloth/Qwen3.8-27B-unsloth-bnb-4bit` |
| L4 など 24GB 級 | `Qwen/Qwen3.8-27B` を読み込み時に NF4 |
| T4 16GB | 27B は載らないので `Qwen/Qwen3.5-9B`（結果は 27B と混ぜない） |

変えたいときは、次のセルのあとで `.env` の `LLM_MODEL_ID` を書き換えてください。


In [ ]:
!python3 -m src gpu-profile --write-env .env

from src.runtime.envfile import update_env_file
from pathlib import Path
import os
update_env_file(Path(".env"), {
    "APP_CONFIG_FILE": "config.colab.yaml",
    "APP_DATA_DIR": os.environ["APP_DATA_DIR"],
    "HF_HOME": os.environ["HF_HOME"],
    "UNIMERNET_MODEL_DIR": os.environ["UNIMERNET_MODEL_DIR"],
    "SERVICE_MODE": "managed",
    "APP_PASSWORD": os.environ["APP_PASSWORD"],
})
print(Path(".env").read_text())


## 4. 画面を開く

次のセルは止めずに動かしたままにします。ログに出る `https://....trycloudflare.com` をブラウザで開き、上で決めたパスワードを入れます。

画面の流れは、セット登録 → OCR実行 → OCR確認 → 診断実行 → 診断結果確認 → 教師採点との比較、です。


In [ ]:
!bash scripts/start_streamlit_tunnel.sh


## 5. 画面を使わず一括で採点する

答案と問題を画面で登録したあと、このセルでまとめて処理できます。モデルのロードは各段階で1回です。
`prob_xxxxxxxx` は登録した問題IDに置き換えてください。


In [ ]:
# 問題IDは画面の「問題登録」に出ます
!python3 -m src batch --problem-id prob_xxxxxxxx


## 6. 教師の採点と比べる

CSV は1行が「生徒 × 設問」です。テンプレートは `data/examples/teacher_grades_template.csv`、または画面から落とせます。
差が満点の 20% 以上の項目を記録し、`--recheck` で採点し直します。元の採点は残ります。


In [ ]:
# Drive に置いた教師採点 CSV のパスに変える
TEACHER_CSV = "/content/drive/MyDrive/answer-correction/teacher_grades.csv"
!python3 -m src import-teacher {TEACHER_CSV} --apply-allocation
!python3 -m src compare --recheck --out /content/drive/MyDrive/answer-correction/comparison.csv


## 7. 終わったら GPU を空ける

セッションを閉じるだけでも解放されます。同じセッションで別の作業をするときは止めてください。


In [ ]:
!python3 -m src services stop
